In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
df_bookings = pd.read_csv(Path(r"C:\Users\krake\Downloads\Build1_DataPack\datapack\bookings\merged_bookings.csv"))
df_collections = pd.read_csv(Path(r"C:\Users\krake\Downloads\Build1_DataPack\datapack\collections\merged_collections.csv"))
df_results = pd.read_csv(Path(r"C:\Users\krake\Downloads\Build1_DataPack\datapack\results\merged_results.csv"))
df_bookings.columns

Index(['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel',
       'source', 'patient_id', 'patient_name', 'test_code', 'test_price',
       'home_fee', 'data_source'],
      dtype='str')

In [2]:
df_bookings_agg = (
    df_bookings.groupby('booking_id')
    .agg(
        created_at=('created_at', 'first'),
        slot_at=('slot_at', 'first'),
        city=('city', 'first'),
        clinic_id=('clinic_id', 'first'),
        channel=('channel', 'first'),
        source=('source', 'first'),
        patient_id=('patient_id', 'first'),
        patient_name=('patient_name', 'first'),
        n_tests=('test_code', 'count'),
        test_revenue=('test_price', 'sum'),
        home_fee=('home_fee', 'first'),
        data_source=('data_source', 'first')
    )
    .reset_index()
)
df_bookings_agg

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,n_tests,test_revenue,home_fee,data_source
0,BK-AHM-202501-000001,2025-01-07 21:46:00+00:00,2025-01-09 02:56:00+00:00,Ahmedabad,AHM-03,Clinic,Walkin,PAHM0028525,Saanvi Bose,3,2000.0,0.0,LEGACY
1,BK-AHM-202501-000002,2025-01-02 23:48:00+00:00,2025-01-06 07:52:00+00:00,Ahmedabad,AHM-04,Clinic,App,PAHM0031384,Nisha Iyer,1,650.0,0.0,LEGACY
2,BK-AHM-202501-000003,2025-01-27 05:16:00+00:00,2025-01-28 08:07:00+00:00,Ahmedabad,AHM-01,Home,App,PAHM0027001,Sneha Menon,4,1670.0,150.0,LEGACY
3,BK-AHM-202501-000004,2025-01-19 21:18:00+00:00,2025-01-24 01:45:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0007069,Arjun Bose,1,160.0,0.0,LEGACY
4,BK-AHM-202501-000005,2025-01-06 02:51:00+00:00,2025-01-11 04:26:00+00:00,Ahmedabad,AHM-03,Clinic,App,PAHM0002520,Ritika Nair,2,1900.0,0.0,LEGACY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
565288,NOVA_JAI_0055749,2026-06-01 23:19:00+00:00,2026-06-08 06:24:00+00:00,Jaipur,JAI-02,Home,App,PJAI0028824,NaN,2,1000.0,150.0,NOVA
565289,NOVA_JAI_0055750,2026-06-15 17:58:00+00:00,2026-06-18 02:52:00+00:00,Jaipur,JAI-01,Home,Walkin,PJAI0028764,NaN,3,1800.0,150.0,NOVA
565290,NOVA_JAI_0055751,2026-06-19 00:06:00+00:00,2026-06-26 06:17:00+00:00,Jaipur,JAI-02,Clinic,Walkin,PJAI0028485,NaN,1,1499.0,0.0,NOVA
565291,NOVA_JAI_0055752,2026-06-06 20:59:00+00:00,2026-06-12 05:48:00+00:00,Jaipur,JAI-02,Home,Phone,PJAI0017296,NaN,3,1320.0,150.0,NOVA


In [3]:
res_by_booking = df_results.groupby('booking_id').agg(n_results=('test_code', 'count'))
bookings = df_bookings_agg.merge(res_by_booking, on ='booking_id', how = 'left')
bookings['n_results'] = bookings['n_results'].fillna(0).astype(int)

In [4]:
dups = df_collections.duplicated(subset=['booking_id', 'status', 'collected_at'], keep='first')
coll = df_collections[~dups]
bookings = bookings.merge(coll[['booking_id', 'status', 'collected_at']],
                          on='booking_id', how='left', validate='one_to_one')
bookings['status'] = bookings['status'].fillna('cancelled_or_missing')

In [5]:
# Filter Jan–Jun 2025 and Jan–Jun 2026

mask = ((bookings['slot_at'].between('2025-01-01', '2025-06-30 23:59:59'))|(bookings['slot_at'].between('2026-01-01', '2026-06-30 23:59:59')))


In [6]:
df = bookings.loc[mask].copy()

In [8]:
# Convert slot_at to datetime
df['slot_at'] = pd.to_datetime(
    df['slot_at'], errors='coerce'
)
# Create half-year labels
df['period'] = df['slot_at'].dt.year
df['month'] = df['slot_at'].dt.month_name

In [9]:
# 1. One row per booking in the selected H1 periods
booking_level = df[
    ['booking_id', 'city', 'period', 'month', 'n_tests', 'status']
].copy()



booking_level = booking_level.rename(
    columns={'n_tests': 'tests_ordered'}
)

# 2. Keep result records only for these bookings
relevant_booking_ids = booking_level['booking_id'].unique()

results_scope = df_results[
    df_results['booking_id'].isin(relevant_booking_ids)
].copy()

# 3. Count result rows per booking
result_level = (
    results_scope.groupby('booking_id')
    .agg(tests_with_results=('test_code', 'count'))
    .reset_index()
)

# 4. Match results to bookings without multiplying booking rows
reconciliation = booking_level.merge(
    result_level,
    on='booking_id',
    how='left',
    validate='one_to_one'
)

reconciliation['tests_with_results'] = (
    reconciliation['tests_with_results']
    .fillna(0)
    .astype(int)
)

# Positive gap = more ordered tests than result rows
reconciliation['result_gap'] = (
    reconciliation['tests_ordered']
    - reconciliation['tests_with_results']
)

reconciliation['has_result_gap'] = (
    reconciliation['result_gap'] > 0
)

reconciliation['results_exceed_orders'] = (
    reconciliation['result_gap'] < 0
)

reconciliation.head()

,booking_id,city,period,month,tests_ordered,status,tests_with_results,result_gap,has_result_gap,results_exceed_orders
0,BK-AHM-202501-000001,Ahmedabad,2025,<bound method DatetimeArray.month_name of <pan...,3,collected,3,0,False,False
1,BK-AHM-202501-000002,Ahmedabad,2025,<bound method DatetimeArray.month_name of <pan...,1,collected,1,0,False,False
2,BK-AHM-202501-000003,Ahmedabad,2025,<bound method DatetimeArray.month_name of <pan...,4,collected,3,1,True,False
3,BK-AHM-202501-000004,Ahmedabad,2025,<bound method DatetimeArray.month_name of <pan...,1,cancelled_or_missing,0,1,True,False
4,BK-AHM-202501-000005,Ahmedabad,2025,<bound method DatetimeArray.month_name of <pan...,2,collected,2,0,False,False


In [10]:
results_summary = (
    reconciliation.groupby(['city', 'period'])
    .agg(
        bookings=('booking_id', 'nunique'),
        tests_ordered=('tests_ordered', 'sum'),
        tests_with_results=('tests_with_results', 'sum'),
        bookings_with_gap=('has_result_gap', 'sum'),
        bookings_with_excess_results=(
            'results_exceed_orders', 'sum'
        ),
        total_result_gap=('result_gap', 'sum')
    )
    .reset_index()
)

results_summary['result_coverage_pct'] = (
    results_summary['tests_with_results']
    / results_summary['tests_ordered'].replace(0, np.nan)
    * 100
).round(2)

results_summary.sort_values(['city', 'period'])

,city,period,bookings,tests_ordered,tests_with_results,bookings_with_gap,bookings_with_excess_results,total_result_gap,result_coverage_pct
0,Ahmedabad,2025,33369,76727,62519,6849,48,14208,81.48
1,Ahmedabad,2026,36220,84816,69371,7293,71,15445,81.79
2,Bengaluru,2025,43618,99732,80977,8961,57,18755,81.19
3,Bengaluru,2026,45678,107355,88263,9060,77,19092,82.22
4,Chennai,2025,34880,80022,64888,7296,46,15134,81.09
5,Chennai,2026,34191,80636,63989,7710,32,16647,79.36
6,Hyderabad,2025,31712,73058,59670,6481,58,13388,81.67
7,Hyderabad,2026,33436,78450,64363,6670,72,14087,82.04
8,Jaipur,2025,19579,45017,36726,4036,38,8291,81.58
9,Jaipur,2026,17168,40164,32121,3757,15,8043,79.97


In [11]:
results_summary = (
    reconciliation.groupby(['city', 'period'])
    .agg(
        bookings=('booking_id', 'nunique'),
        tests_ordered=('tests_ordered', 'sum'),
        tests_with_results=('tests_with_results', 'sum'),
        bookings_with_gap=('has_result_gap', 'sum'),
        bookings_with_excess_results=(
            'results_exceed_orders', 'sum'
        ),
        total_result_gap=('result_gap', 'sum')
    )
    .reset_index()
)

results_summary['result_coverage_pct'] = (
    results_summary['tests_with_results']
    / results_summary['tests_ordered'].replace(0, np.nan)
    * 100
).round(2)

results_summary.sort_values(['city', 'period'])

,city,period,bookings,tests_ordered,tests_with_results,bookings_with_gap,bookings_with_excess_results,total_result_gap,result_coverage_pct
0,Ahmedabad,2025,33369,76727,62519,6849,48,14208,81.48
1,Ahmedabad,2026,36220,84816,69371,7293,71,15445,81.79
2,Bengaluru,2025,43618,99732,80977,8961,57,18755,81.19
3,Bengaluru,2026,45678,107355,88263,9060,77,19092,82.22
4,Chennai,2025,34880,80022,64888,7296,46,15134,81.09
5,Chennai,2026,34191,80636,63989,7710,32,16647,79.36
6,Hyderabad,2025,31712,73058,59670,6481,58,13388,81.67
7,Hyderabad,2026,33436,78450,64363,6670,72,14087,82.04
8,Jaipur,2025,19579,45017,36726,4036,38,8291,81.58
9,Jaipur,2026,17168,40164,32121,3757,15,8043,79.97


In [12]:
results_monthly = (
    reconciliation.groupby(['city', 'month', 'period'])
    .agg(
        tests_ordered=('tests_ordered', 'sum'),
        tests_with_results=('tests_with_results', 'sum'),
        result_gap=('result_gap', 'sum'),
        bookings=('booking_id', 'nunique'),
        bookings_with_gap=('has_result_gap', 'sum')
    )
    .reset_index()
)

results_monthly['result_coverage_pct'] = (
    results_monthly['tests_with_results']
    / results_monthly['tests_ordered'].replace(0, np.nan)
    * 100
).round(2)

results_monthly.sort_values(
    ['city', 'month', 'period']
)

,city,month,period,tests_ordered,tests_with_results,result_gap,bookings,bookings_with_gap,result_coverage_pct
0,Ahmedabad,<bound method DatetimeArray.month_name of <pan...,2025,76727,62519,14208,33369,6849,81.48
1,Ahmedabad,<bound method DatetimeArray.month_name of <pan...,2026,84816,69371,15445,36220,7293,81.79
2,Bengaluru,<bound method DatetimeArray.month_name of <pan...,2025,99732,80977,18755,43618,8961,81.19
3,Bengaluru,<bound method DatetimeArray.month_name of <pan...,2026,107355,88263,19092,45678,9060,82.22
4,Chennai,<bound method DatetimeArray.month_name of <pan...,2025,80022,64888,15134,34880,7296,81.09
5,Chennai,<bound method DatetimeArray.month_name of <pan...,2026,80636,63989,16647,34191,7710,79.36
6,Hyderabad,<bound method DatetimeArray.month_name of <pan...,2025,73058,59670,13388,31712,6481,81.67
7,Hyderabad,<bound method DatetimeArray.month_name of <pan...,2026,78450,64363,14087,33436,6670,82.04
8,Jaipur,<bound method DatetimeArray.month_name of <pan...,2025,45017,36726,8291,19579,4036,81.58
9,Jaipur,<bound method DatetimeArray.month_name of <pan...,2026,40164,32121,8043,17168,3757,79.97


In [13]:
duplicate_results = (
    results_scope.groupby(['booking_id', 'test_code'])
    .size()
    .reset_index(name='row_count')
    .query('row_count > 1')
    .sort_values('row_count', ascending=False)
)

duplicate_results.head(20)

,booking_id,test_code,row_count


In [14]:
status_gap = (
    reconciliation.groupby(['period', 'status'])
    .agg(
        bookings=('booking_id', 'nunique'),
        tests_ordered=('tests_ordered', 'sum'),
        tests_with_results=('tests_with_results', 'sum'),
        result_gap=('result_gap', 'sum')
    )
    .reset_index()
)

status_gap['result_coverage_pct'] = (
    status_gap['tests_with_results']
    / status_gap['tests_ordered'].replace(0, np.nan)
    * 100
).round(2)

status_gap.sort_values(['period', 'result_gap'], ascending=[True, False])

,period,status,bookings,tests_ordered,tests_with_results,result_gap,result_coverage_pct
2,2025,no_show,26817,61676,0,61676,0.00
0,2025,cancelled_or_missing,5669,12973,0,12973,0.00
1,2025,collected,156959,360359,354137,6222,98.27
5,2026,no_show,26247,61560,0,61560,0.00
3,2026,cancelled_or_missing,7223,17072,0,17072,0.00
4,2026,collected,161194,378156,371742,6414,98.30


In [15]:
largest_gaps = (
    reconciliation[reconciliation['result_gap'] > 0]
    .sort_values('result_gap', ascending=False)
)

largest_gaps[
    [
        'booking_id', 'city', 'period', 'month', 'status',
        'tests_ordered', 'tests_with_results', 'result_gap'
    ]
].head(20)

,booking_id,city,period,month,status,tests_ordered,tests_with_results,result_gap
40432,BK-AHM-202602-072364,Ahmedabad,2026,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
373994,NOVA_CHE_0103326,Chennai,2026,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
263277,BK-HYD-202604-082067,Hyderabad,2026,<bound method DatetimeArray.month_name of <pan...,cancelled_or_missing,6,0,6
43870,BK-AHM-202602-075807,Ahmedabad,2026,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
234167,BK-HYD-202505-022835,Hyderabad,2025,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
297368,BK-JAI-202601-039316,Jaipur,2026,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
286904,BK-JAI-202503-010408,Jaipur,2025,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
67602,BK-AHM-202606-099595,Ahmedabad,2026,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
312828,BK-PUN-202502-007985,Pune,2025,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
320716,BK-PUN-202504-015893,Pune,2025,<bound method DatetimeArray.month_name of <pan...,no_show,6,0,6
